# CH-SIMS D1 - D2 

In [17]:
"""
CH-SIMS: Neural Network with Late Fusion (2-branch)
- Text: Dense layers
- Vision: Dense layers
- Late Fusion: Average
"""

import pickle
import time
import numpy as np
from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import accuracy_score, classification_report, f1_score

start_time = time.time()

# ============================================================
# 1. Load CH-SIMS Data
# ============================================================
file_path = 'SIMS/Processed/unaligned_39.pkl'
with open(file_path, 'rb') as f:
    data = pickle.load(f)

train = data['train']
valid = data['valid']
test = data['test']

def aggregate_mean(X):
    return np.mean(X, axis=1)

X_text_train = aggregate_mean(train['text'])
X_vision_train = aggregate_mean(train['vision'])
y_train = train['classification_labels'].astype(int)

X_text_val = aggregate_mean(valid['text'])
X_vision_val = aggregate_mean(valid['vision'])
y_val = valid['classification_labels'].astype(int)

X_text_test = aggregate_mean(test['text'])
X_vision_test = aggregate_mean(test['vision'])
y_test = test['classification_labels'].astype(int)

print("=" * 60)
print("CH-SIMS Data Shapes")
print("=" * 60)
print(f"Train: Text {X_text_train.shape}, Vision {X_vision_train.shape}")
print(f"Val:   Text {X_text_val.shape}, Vision {X_vision_val.shape}")
print(f"Test:  Text {X_text_test.shape}, Vision {X_vision_test.shape}")
print(f"Labels (train): {np.bincount(y_train)}")
print("=" * 60)

# ============================================================
# 2. Convert labels to one-hot
# ============================================================
num_classes = 3
y_train_cat = to_categorical(y_train, num_classes)
y_val_cat = to_categorical(y_val, num_classes)
y_test_cat = to_categorical(y_test, num_classes)

print(f"y_train_cat shape: {y_train_cat.shape}")

# ============================================================
# 3. Build Model
# ============================================================

# ----- Text branch -----
text_input = Input(shape=(X_text_train.shape[1],), name='text_input')
t = Dense(128, activation='relu')(text_input)
t = BatchNormalization()(t)
t = Dropout(0.3)(t)
t = Dense(64, activation='relu')(t)
t = Dropout(0.3)(t)
text_out = Dense(num_classes, activation='softmax', name='text_softmax')(t)

# ----- Vision branch -----
vision_input = Input(shape=(X_vision_train.shape[1],), name='vision_input')
v = Dense(128, activation='relu')(vision_input)
v = BatchNormalization()(v)
v = Dropout(0.3)(v)
v = Dense(64, activation='relu')(v)
v = Dropout(0.3)(v)
vision_out = Dense(num_classes, activation='softmax', name='vision_softmax')(v)

# ----- Define model -----
model = Model(
    inputs=[text_input, vision_input],
    outputs=[text_out, vision_out],
)

model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

print("\nModel Summary:")
model.summary()

# ============================================================
# 4. Training
# ============================================================
print("\nTraining Model...")
history = model.fit(
    [X_text_train, X_vision_train],
    [y_train_cat, y_train_cat],
    validation_data=([X_text_val, X_vision_val], [y_val_cat, y_val_cat]),
    epochs=30,
    batch_size=64,
    verbose=1
)

# ============================================================
# 5. Prediction & Late Fusion (Average)
# ============================================================
print("\nPredicting...")
text_probs, vision_probs = model.predict([X_text_test, X_vision_test])

# Late Fusion: Average
ensemble_probs = (text_probs + vision_probs) / 2.0
predictions = np.argmax(ensemble_probs, axis=1)
true_labels = y_test

# ============================================================
# 6. Evaluation
# ============================================================
print("\n" + "=" * 60)
print("CH-SIMS Results with Neural Network + Late Fusion (Average)")
print("=" * 60)
print(f"Test Accuracy: {accuracy_score(true_labels, predictions):.4f}")
print(f"Test Weighted F1: {f1_score(true_labels, predictions, average='weighted'):.4f}")
print("\nTest Classification Report:")
print(classification_report(
    true_labels, predictions, digits=4,
    target_names=['negative', 'neutral', 'positive']
))

# ============================================================
# 7. Unimodal Baselines
# ============================================================
print("\n" + "=" * 60)
print("Unimodal Baselines")
print("=" * 60)

text_pred = np.argmax(text_probs, axis=1)
vision_pred = np.argmax(vision_probs, axis=1)

print(f"Text-only Accuracy: {accuracy_score(true_labels, text_pred):.4f}")
print(f"Text-only Weighted F1: {f1_score(true_labels, text_pred, average='weighted'):.4f}")
print(f"Vision-only Accuracy: {accuracy_score(true_labels, vision_pred):.4f}")
print(f"Vision-only Weighted F1: {f1_score(true_labels, vision_pred, average='weighted'):.4f}")

# ============================================================
# 8. Runtime
# ============================================================
end_time = time.time()
elapsed_time = end_time - start_time
print(f"\nTotal Execution Time: {elapsed_time:.4f} seconds")

CH-SIMS Data Shapes
Train: Text (1368, 768), Vision (1368, 709)
Val:   Text (456, 768), Vision (456, 709)
Test:  Text (457, 768), Vision (457, 709)
Labels (train): [742 207 419]
y_train_cat shape: (1368, 3)

Model Summary:
Model: "model_20"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 text_input (InputLayer)     [(None, 768)]                0         []                            
                                                                                                  
 vision_input (InputLayer)   [(None, 709)]                0         []                            
                                                                                                  
 dense_91 (Dense)            (None, 128)                  98432     ['text_input[0][0]']          
                                                                  

Epoch 12/30
22/22 [==============================] - 0s 5ms/step - loss: 1.4534 - text_softmax_loss: 0.4286 - vision_softmax_loss: 1.0248 - text_softmax_accuracy: 0.8399 - vision_softmax_accuracy: 0.5365 - val_loss: 1.8842 - val_text_softmax_loss: 0.8805 - val_vision_softmax_loss: 1.0037 - val_text_softmax_accuracy: 0.6491 - val_vision_softmax_accuracy: 0.5439
Epoch 13/30
22/22 [==============================] - 0s 5ms/step - loss: 1.4163 - text_softmax_loss: 0.3954 - vision_softmax_loss: 1.0209 - text_softmax_accuracy: 0.8421 - vision_softmax_accuracy: 0.5205 - val_loss: 1.9032 - val_text_softmax_loss: 0.9131 - val_vision_softmax_loss: 0.9901 - val_text_softmax_accuracy: 0.6360 - val_vision_softmax_accuracy: 0.5307
Epoch 14/30
22/22 [==============================] - 0s 5ms/step - loss: 1.3969 - text_softmax_loss: 0.3836 - vision_softmax_loss: 1.0133 - text_softmax_accuracy: 0.8458 - vision_softmax_accuracy: 0.5175 - val_loss: 1.8880 - val_text_softmax_loss: 0.9127 - val_vision_softma

In [ ]:
============================================================
CH-SIMS Binary Classification
============================================================
Train: Text (1161, 768), Vision (1161, 709)
Labels (train): [742 419]
Labels (test): [248 140]
============================================================

Training ExtraTree Text (Binary)...
Training ExtraTree Vision (Binary)...

Applying DPF Fusion (β=1.1)...

Training XGBoost Meta-Classifier...

============================================================
CH-SIMS RESULTS
C:\Users\ausco\anaconda3\lib\site-packages\xgboost\sklearn.py:1395: UserWarning: `use_label_encoder` is deprecated in 1.7.0.
  warnings.warn("`use_label_encoder` is deprecated in 1.7.0.")
============================================================
Accuracy:  0.7912
Weighted F1: 0.7884

Classification Report:
              precision    recall  f1-score   support

    negative     0.8175    0.8669    0.8415       248
    positive     0.7360    0.6571    0.6943       140

    accuracy                         0.7912       388
   macro avg     0.7767    0.7620    0.7679       388
weighted avg     0.7881    0.7912    0.7884       388


============================================================
BASELINES
============================================================

Method                    Accuracy     Weighted F1 
--------------------------------------------------
DPF (β=1.1)               0.7912      0.7884
Uniform (β=0)             0.7655      0.7403
Text-only                 0.7655      0.7403
Vision-only               0.7139      0.7094
============================================================


In [20]:
"""
CH-SIMS: Neural Network with Late Fusion (Binary Classification)
Paper-Ready Version - Uses Weighted F1
"""

import pickle
import time
import numpy as np
from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import accuracy_score, f1_score, classification_report

start_time = time.time()

# ============================================================
# 1. Load CH-SIMS Data
# ============================================================
file_path = 'SIMS/Processed/unaligned_39.pkl'
with open(file_path, 'rb') as f:
    data = pickle.load(f)

train = data['train']
valid = data['valid']
test = data['test']

def aggregate_mean(X):
    return np.mean(X, axis=1)

X_text_train = aggregate_mean(train['text'])
X_vision_train = aggregate_mean(train['vision'])
y_train = train['classification_labels'].astype(int)

X_text_val = aggregate_mean(valid['text'])
X_vision_val = aggregate_mean(valid['vision'])
y_val = valid['classification_labels'].astype(int)

X_text_test = aggregate_mean(test['text'])
X_vision_test = aggregate_mean(test['vision'])
y_test = test['classification_labels'].astype(int)

# ============================================================
# 2. Convert to Binary Classification (Remove Neutral)
# ============================================================
def convert_to_binary(X_text, X_vision, y):
    mask = (y != 1)
    X_text_bin = X_text[mask]
    X_vision_bin = X_vision[mask]
    y_bin = y[mask]
    y_bin = np.where(y_bin == 2, 1, 0)
    return X_text_bin, X_vision_bin, y_bin

X_text_train_bin, X_vision_train_bin, y_train_bin = convert_to_binary(
    X_text_train, X_vision_train, y_train
)
X_text_val_bin, X_vision_val_bin, y_val_bin = convert_to_binary(
    X_text_val, X_vision_val, y_val
)
X_text_test_bin, X_vision_test_bin, y_test_bin = convert_to_binary(
    X_text_test, X_vision_test, y_test
)

print("=" * 60)
print("CH-SIMS Binary Classification (Neural Network)")
print("=" * 60)
print(f"Train: Text {X_text_train_bin.shape}, Vision {X_vision_train_bin.shape}")
print(f"Labels (train): {np.bincount(y_train_bin)}")
print(f"Labels (test): {np.bincount(y_test_bin)}")
print("=" * 60)

# ============================================================
# 3. Convert labels to one-hot
# ============================================================
num_classes = 2
y_train_cat = to_categorical(y_train_bin, num_classes)
y_val_cat = to_categorical(y_val_bin, num_classes)
y_test_cat = to_categorical(y_test_bin, num_classes)

# ============================================================
# 4. Build Model (2 branches for late fusion)
# ============================================================

# ----- Text branch -----
text_input = Input(shape=(X_text_train_bin.shape[1],), name='text_input')
t = Dense(128, activation='relu')(text_input)
t = BatchNormalization()(t)
t = Dropout(0.3)(t)
t = Dense(64, activation='relu')(t)
t = Dropout(0.3)(t)
text_out = Dense(num_classes, activation='softmax', name='text_softmax')(t)

# ----- Vision branch -----
vision_input = Input(shape=(X_vision_train_bin.shape[1],), name='vision_input')
v = Dense(128, activation='relu')(vision_input)
v = BatchNormalization()(v)
v = Dropout(0.3)(v)
v = Dense(64, activation='relu')(v)
v = Dropout(0.3)(v)
vision_out = Dense(num_classes, activation='softmax', name='vision_softmax')(v)

# ----- Define model -----
model = Model(
    inputs=[text_input, vision_input],
    outputs=[text_out, vision_out]
)

model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

print("\nModel Summary:")
model.summary()

# ============================================================
# 5. Training
# ============================================================
print("\nTraining Model...")
history = model.fit(
    [X_text_train_bin, X_vision_train_bin],
    [y_train_cat, y_train_cat],
    validation_data=([X_text_val_bin, X_vision_val_bin], [y_val_cat, y_val_cat]),
    epochs=10,
    batch_size=64,
    verbose=1
)

# ============================================================
# 6. Prediction & Late Fusion (Average)
# ============================================================
print("\nPredicting...")
text_probs, vision_probs = model.predict([X_text_test_bin, X_vision_test_bin])

# Late Fusion: Average
ensemble_probs = (text_probs + vision_probs) / 2.0
predictions = np.argmax(ensemble_probs, axis=1)
true_labels = y_test_bin

# ============================================================
# 7. Evaluation (Weighted F1)
# ============================================================
print("\n" + "=" * 60)
print("CH-SIMS RESULTS (Neural Network + Late Fusion)")
print("=" * 60)
print(f"Test Accuracy: {accuracy_score(true_labels, predictions):.4f}")
print(f"Test Weighted F1: {f1_score(true_labels, predictions, average='weighted'):.4f}")
print("\nTest Classification Report:")
print(classification_report(
    true_labels, predictions, digits=4,
    target_names=['negative', 'positive']
))

# ============================================================
# 8. Unimodal Baselines
# ============================================================
print("\n" + "=" * 60)
print("UNIMODAL BASELINES")
print("=" * 60)

text_pred = np.argmax(text_probs, axis=1)
vision_pred = np.argmax(vision_probs, axis=1)

print(f"Text-only Accuracy: {accuracy_score(true_labels, text_pred):.4f}")
print(f"Text-only Weighted F1: {f1_score(true_labels, text_pred, average='weighted'):.4f}")
print(f"Vision-only Accuracy: {accuracy_score(true_labels, vision_pred):.4f}")
print(f"Vision-only Weighted F1: {f1_score(true_labels, vision_pred, average='weighted'):.4f}")

# ============================================================
# 9. Summary Table
# ============================================================
print("\n" + "=" * 60)
print("CH-SIMS SUMMARY TABLE (Neural Network)")
print("=" * 60)
print(f"{'Method':<25} {'Accuracy':<12} {'Weighted F1':<12}")
print("-" * 50)
print(f"{'Late Fusion (Avg)':<25} {accuracy_score(true_labels, predictions):.4f}      {f1_score(true_labels, predictions, average='weighted'):.4f}")
print(f"{'Text-only (NN)':<25} {accuracy_score(true_labels, text_pred):.4f}      {f1_score(true_labels, text_pred, average='weighted'):.4f}")
print(f"{'Vision-only (NN)':<25} {accuracy_score(true_labels, vision_pred):.4f}      {f1_score(true_labels, vision_pred, average='weighted'):.4f}")
print("=" * 60)

# ============================================================
# 10. Runtime
# ============================================================
end_time = time.time()
elapsed_time = end_time - start_time
print(f"\nTotal Execution Time: {elapsed_time:.4f} seconds")

CH-SIMS Binary Classification (Neural Network)
Train: Text (1161, 768), Vision (1161, 709)
Labels (train): [742 419]
Labels (test): [248 140]

Model Summary:
Model: "model_21"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 text_input (InputLayer)     [(None, 768)]                0         []                            
                                                                                                  
 vision_input (InputLayer)   [(None, 709)]                0         []                            
                                                                                                  
 dense_95 (Dense)            (None, 128)                  98432     ['text_input[0][0]']          
                                                                                                  
 dense_97 (Dense)            (No

# Late Fusion

In [22]:
"""
CH-SIMS: Neural Network with Early Fusion (Binary Classification)
Paper-Ready Version - Uses Weighted F1
"""

import pickle
import time
import numpy as np
from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization, Concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import legacy
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import accuracy_score, f1_score, classification_report

start_time = time.time()

# ============================================================
# 1. Load CH-SIMS Data
# ============================================================
file_path = 'SIMS/Processed/unaligned_39.pkl'
with open(file_path, 'rb') as f:
    data = pickle.load(f)

train = data['train']
valid = data['valid']
test = data['test']

def aggregate_mean(X):
    return np.mean(X, axis=1)

X_text_train = aggregate_mean(train['text'])
X_vision_train = aggregate_mean(train['vision'])
y_train = train['classification_labels'].astype(int)

X_text_val = aggregate_mean(valid['text'])
X_vision_val = aggregate_mean(valid['vision'])
y_val = valid['classification_labels'].astype(int)

X_text_test = aggregate_mean(test['text'])
X_vision_test = aggregate_mean(test['vision'])
y_test = test['classification_labels'].astype(int)

# ============================================================
# 2. Convert to Binary Classification (Remove Neutral)
# ============================================================
def convert_to_binary(X_text, X_vision, y):
    mask = (y != 1)
    X_text_bin = X_text[mask]
    X_vision_bin = X_vision[mask]
    y_bin = y[mask]
    y_bin = np.where(y_bin == 2, 1, 0)
    return X_text_bin, X_vision_bin, y_bin

X_text_train_bin, X_vision_train_bin, y_train_bin = convert_to_binary(
    X_text_train, X_vision_train, y_train
)
X_text_val_bin, X_vision_val_bin, y_val_bin = convert_to_binary(
    X_text_val, X_vision_val, y_val
)
X_text_test_bin, X_vision_test_bin, y_test_bin = convert_to_binary(
    X_text_test, X_vision_test, y_test
)

print("=" * 60)
print("CH-SIMS Binary Classification (Early Fusion - Neural Network)")
print("=" * 60)
print(f"Train: Text {X_text_train_bin.shape}, Vision {X_vision_train_bin.shape}")
print(f"Labels (train): {np.bincount(y_train_bin)}")
print(f"Labels (test): {np.bincount(y_test_bin)}")
print("=" * 60)

# ============================================================
# 3. Convert labels to one-hot
# ============================================================
num_classes = 2
y_train_cat = to_categorical(y_train_bin, num_classes)
y_val_cat = to_categorical(y_val_bin, num_classes)
y_test_cat = to_categorical(y_test_bin, num_classes)

# ============================================================
# 4. Build Models
# ============================================================

# ----- Build Early Fusion Model -----
def build_early_fusion_model(text_dim, vision_dim, num_classes):
    # Text branch
    text_input = Input(shape=(text_dim,), name='text_input')
    t = Dense(128, activation='relu')(text_input)
    t = BatchNormalization()(t)
    t = Dropout(0.3)(t)
    t = Dense(64, activation='relu')(t)
    t = Dropout(0.3)(t)
    
    # Vision branch
    vision_input = Input(shape=(vision_dim,), name='vision_input')
    v = Dense(128, activation='relu')(vision_input)
    v = BatchNormalization()(v)
    v = Dropout(0.3)(v)
    v = Dense(64, activation='relu')(v)
    v = Dropout(0.3)(v)
    
    # Early Fusion: Concatenate
    fusion = Concatenate()([t, v])
    
    # Fusion layers
    fusion = Dense(128, activation='relu')(fusion)
    fusion = Dropout(0.5)(fusion)
    fusion = Dense(64, activation='relu')(fusion)
    fusion = Dropout(0.3)(fusion)
    output = Dense(num_classes, activation='softmax')(fusion)
    
    model = Model(inputs=[text_input, vision_input], outputs=output)
    return model

# ----- Build Unimodal Models (for baselines) -----
def build_unimodal_model(dim, num_classes, modality_name='text'):
    inputs = Input(shape=(dim,), name=f'{modality_name}_input')
    x = Dense(128, activation='relu')(inputs)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.3)(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.3)(x)
    output = Dense(num_classes, activation='softmax')(x)
    
    model = Model(inputs=inputs, outputs=output)
    return model

# ============================================================
# 5. Build and Compile Models (使用 legacy Adam)
# ============================================================

# Early Fusion Model
fusion_model = build_early_fusion_model(
    X_text_train_bin.shape[1],
    X_vision_train_bin.shape[1],
    num_classes
)

optimizer = legacy.Adam(learning_rate=0.001)  # ← 使用 legacy.Adam
fusion_model.compile(
    loss='categorical_crossentropy',
    optimizer=optimizer,
    metrics=['accuracy']
)

print("\nEarly Fusion Model Summary:")
fusion_model.summary()

# Text-only Model
text_model = build_unimodal_model(X_text_train_bin.shape[1], num_classes, 'text')
text_model.compile(
    loss='categorical_crossentropy',
    optimizer=legacy.Adam(learning_rate=0.001),  # ← 使用 legacy.Adam
    metrics=['accuracy']
)

# Vision-only Model
vision_model = build_unimodal_model(X_vision_train_bin.shape[1], num_classes, 'vision')
vision_model.compile(
    loss='categorical_crossentropy',
    optimizer=legacy.Adam(learning_rate=0.001),  # ← 使用 legacy.Adam
    metrics=['accuracy']
)

# ============================================================
# 6. Training
# ============================================================
print("\n" + "=" * 60)
print("Training Early Fusion Model...")
print("=" * 60)

fusion_history = fusion_model.fit(
    [X_text_train_bin, X_vision_train_bin],
    y_train_cat,
    validation_data=([X_text_val_bin, X_vision_val_bin], y_val_cat),
    epochs=10,
    batch_size=64,
    verbose=1
)

print("\n" + "=" * 60)
print("Training Text-only Model...")
print("=" * 60)

text_history = text_model.fit(
    X_text_train_bin,
    y_train_cat,
    validation_data=(X_text_val_bin, y_val_cat),
    epochs=10,
    batch_size=32,
    verbose=1
)

print("\n" + "=" * 60)
print("Training Vision-only Model...")
print("=" * 60)

vision_history = vision_model.fit(
    X_vision_train_bin,
    y_train_cat,
    validation_data=(X_vision_val_bin, y_val_cat),
    epochs=30,
    batch_size=32,
    verbose=1
)

# ============================================================
# 7. Prediction & Evaluation
# ============================================================
print("\n" + "=" * 60)
print("CH-SIMS RESULTS")
print("=" * 60)

# Early Fusion
fusion_probs = fusion_model.predict([X_text_test_bin, X_vision_test_bin])
fusion_pred = np.argmax(fusion_probs, axis=1)
true_labels = y_test_bin

print("\n--- Early Fusion ---")
print(f"Accuracy:  {accuracy_score(true_labels, fusion_pred):.4f}")
print(f"Weighted F1: {f1_score(true_labels, fusion_pred, average='weighted'):.4f}")
print("\nClassification Report:")
print(classification_report(
    true_labels, fusion_pred, digits=4,
    target_names=['negative', 'positive']
))

# Text-only
text_probs = text_model.predict(X_text_test_bin)
text_pred = np.argmax(text_probs, axis=1)

print("\n--- Text-only ---")
print(f"Accuracy:  {accuracy_score(true_labels, text_pred):.4f}")
print(f"Weighted F1: {f1_score(true_labels, text_pred, average='weighted'):.4f}")

# Vision-only
vision_probs = vision_model.predict(X_vision_test_bin)
vision_pred = np.argmax(vision_probs, axis=1)

print("\n--- Vision-only ---")
print(f"Accuracy:  {accuracy_score(true_labels, vision_pred):.4f}")
print(f"Weighted F1: {f1_score(true_labels, vision_pred, average='weighted'):.4f}")

# ============================================================
# 8. Summary Table
# ============================================================
print("\n" + "=" * 60)
print("CH-SIMS SUMMARY TABLE")
print("=" * 60)
print(f"{'Method':<30} {'Accuracy':<12} {'Weighted F1':<12}")
print("-" * 55)
print(f"{'Early Fusion (NN)':<30} {accuracy_score(true_labels, fusion_pred):.4f}      {f1_score(true_labels, fusion_pred, average='weighted'):.4f}")
print(f"{'Text-only (NN)':<30} {accuracy_score(true_labels, text_pred):.4f}      {f1_score(true_labels, text_pred, average='weighted'):.4f}")
print(f"{'Vision-only (NN)':<30} {accuracy_score(true_labels, vision_pred):.4f}      {f1_score(true_labels, vision_pred, average='weighted'):.4f}")
print("=" * 60)

# ============================================================
# 9. Runtime
# ============================================================
end_time = time.time()
elapsed_time = end_time - start_time
print(f"\nTotal Execution Time: {elapsed_time:.4f} seconds")

CH-SIMS Binary Classification (Early Fusion - Neural Network)
Train: Text (1161, 768), Vision (1161, 709)
Labels (train): [742 419]
Labels (test): [248 140]

Early Fusion Model Summary:
Model: "model_25"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 text_input (InputLayer)     [(None, 768)]                0         []                            
                                                                                                  
 vision_input (InputLayer)   [(None, 709)]                0         []                            
                                                                                                  
 dense_114 (Dense)           (None, 128)                  98432     ['text_input[0][0]']          
                                                                                                  
 den

Epoch 1/30
37/37 [==============================] - 1s 7ms/step - loss: 0.8106 - accuracy: 0.5685 - val_loss: 0.7770 - val_accuracy: 0.6408
Epoch 2/30
37/37 [==============================] - 0s 3ms/step - loss: 0.7180 - accuracy: 0.5866 - val_loss: 0.6871 - val_accuracy: 0.6098
Epoch 3/30
37/37 [==============================] - 0s 3ms/step - loss: 0.7003 - accuracy: 0.6047 - val_loss: 0.6776 - val_accuracy: 0.6098
Epoch 4/30
37/37 [==============================] - 0s 3ms/step - loss: 0.7006 - accuracy: 0.6047 - val_loss: 0.6621 - val_accuracy: 0.6408
Epoch 5/30
37/37 [==============================] - 0s 3ms/step - loss: 0.6837 - accuracy: 0.6124 - val_loss: 0.6574 - val_accuracy: 0.6408
Epoch 6/30
37/37 [==============================] - 0s 3ms/step - loss: 0.6936 - accuracy: 0.6141 - val_loss: 0.6560 - val_accuracy: 0.6408
Epoch 7/30
37/37 [==============================] - 0s 3ms/step - loss: 0.6950 - accuracy: 0.6098 - val_loss: 0.6554 - val_accuracy: 0.6408
Epoch 8/30
37/37 [==

# MultiHeadAttention Transformer +MLP

# D3-Late Fusion

In [10]:
"""
CH-SIMS: Transformer for Text + Dense for Vision (Binary Classification)
Paper-Ready Version - 10 Epochs for Fair Comparison
"""

import pickle
import time
import numpy as np
from tensorflow.keras.layers import (Input, Dense, Dropout, BatchNormalization, 
                                     MultiHeadAttention, LayerNormalization,
                                     GlobalAveragePooling1D, Concatenate, Reshape)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import legacy
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import accuracy_score, f1_score, classification_report

start_time = time.time()

# ============================================================
# 1. Load CH-SIMS Data
# ============================================================
file_path = 'SIMS/Processed/unaligned_39.pkl'
with open(file_path, 'rb') as f:
    data = pickle.load(f)

train = data['train']
valid = data['valid']
test = data['test']

def aggregate_mean(X):
    return np.mean(X, axis=1)

X_text_train = aggregate_mean(train['text'])
X_vision_train = aggregate_mean(train['vision'])
y_train = train['classification_labels'].astype(int)

X_text_val = aggregate_mean(valid['text'])
X_vision_val = aggregate_mean(valid['vision'])
y_val = valid['classification_labels'].astype(int)

X_text_test = aggregate_mean(test['text'])
X_vision_test = aggregate_mean(test['vision'])
y_test = test['classification_labels'].astype(int)

# ============================================================
# 2. Convert to Binary Classification (Remove Neutral)
# ============================================================
def convert_to_binary(X_text, X_vision, y):
    mask = (y != 1)
    X_text_bin = X_text[mask]
    X_vision_bin = X_vision[mask]
    y_bin = y[mask]
    y_bin = np.where(y_bin == 2, 1, 0)
    return X_text_bin, X_vision_bin, y_bin

X_text_train_bin, X_vision_train_bin, y_train_bin = convert_to_binary(
    X_text_train, X_vision_train, y_train
)
X_text_val_bin, X_vision_val_bin, y_val_bin = convert_to_binary(
    X_text_val, X_vision_val, y_val
)
X_text_test_bin, X_vision_test_bin, y_test_bin = convert_to_binary(
    X_text_test, X_vision_test, y_test
)

print("=" * 60)
print("CH-SIMS Binary Classification (Transformer + Dense)")
print("=" * 60)
print(f"Train: Text {X_text_train_bin.shape}, Vision {X_vision_train_bin.shape}")
print(f"Labels (train): {np.bincount(y_train_bin)}")
print(f"Labels (test): {np.bincount(y_test_bin)}")
print("=" * 60)

# ============================================================
# 3. Convert labels to one-hot
# ============================================================
num_classes = 2
y_train_cat = to_categorical(y_train_bin, num_classes)
y_val_cat = to_categorical(y_val_bin, num_classes)
y_test_cat = to_categorical(y_test_bin, num_classes)

# ============================================================
# 4. Build Models
# ============================================================

# ----- Text Branch: Transformer -----
def build_text_transformer(input_dim):
    text_input = Input(shape=(input_dim,), name='text_input')
    
    x = Reshape((input_dim, 1))(text_input)
    attn = MultiHeadAttention(num_heads=4, key_dim=32)(x, x)
    attn = LayerNormalization()(attn + x)
    attn = GlobalAveragePooling1D()(attn)
    
    x = Dense(128, activation='relu')(attn)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.3)(x)
    
    text_out = Dense(num_classes, activation='softmax', name='text_softmax')(x)
    
    return text_input, text_out

# ----- Vision Branch: Dense -----
def build_vision_dense(input_dim):
    vision_input = Input(shape=(input_dim,), name='vision_input')
    
    x = Dense(128, activation='relu')(vision_input)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.3)(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.3)(x)
    
    vision_out = Dense(num_classes, activation='softmax', name='vision_softmax')(x)
    
    return vision_input, vision_out

# ----- Build Model -----
text_input, text_out = build_text_transformer(X_text_train_bin.shape[1])
vision_input, vision_out = build_vision_dense(X_vision_train_bin.shape[1])

model = Model(
    inputs=[text_input, vision_input],
    outputs=[text_out, vision_out]
)

# ============================================================
# 5. Compile Model
# ============================================================
optimizer = legacy.Adam(learning_rate=0.001)
model.compile(
    loss='categorical_crossentropy',
    optimizer=optimizer,
    loss_weights={'text_softmax': 1.0, 'vision_softmax': 1.0},
    metrics={'text_softmax': ['accuracy'], 'vision_softmax': ['accuracy']}
)

print("\nModel Summary:")
model.summary()

# ============================================================
# 6. Training (10 Epochs for Fair Comparison)
# ============================================================
EPOCHS = 10
BATCH_SIZE = 64

print("\n" + "=" * 60)
print(f"Training Model ({EPOCHS} epochs, batch_size={BATCH_SIZE})...")
print("=" * 60)

history = model.fit(
    [X_text_train_bin, X_vision_train_bin],
    [y_train_cat, y_train_cat],
    validation_data=([X_text_val_bin, X_vision_val_bin], [y_val_cat, y_val_cat]),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1
)

# ============================================================
# 7. Prediction & Late Fusion (Average)
# ============================================================
print("\nPredicting...")
text_probs, vision_probs = model.predict([X_text_test_bin, X_vision_test_bin])

# Late Fusion: Average
ensemble_probs = (text_probs + vision_probs) / 2.0
predictions = np.argmax(ensemble_probs, axis=1)
true_labels = y_test_bin

# ============================================================
# 8. Evaluation (Weighted F1)
# ============================================================
print("\n" + "=" * 60)
print("CH-SIMS RESULTS (Transformer + Dense + Late Fusion)")
print("=" * 60)
print(f"Test Accuracy: {accuracy_score(true_labels, predictions):.4f}")
print(f"Test Weighted F1: {f1_score(true_labels, predictions, average='weighted'):.4f}")
print("\nTest Classification Report:")
print(classification_report(
    true_labels, predictions, digits=4,
    target_names=['negative', 'positive']
))

# ============================================================
# 9. Unimodal Baselines
# ============================================================
print("\n" + "=" * 60)
print("UNIMODAL BASELINES")
print("=" * 60)

text_pred = np.argmax(text_probs, axis=1)
vision_pred = np.argmax(vision_probs, axis=1)

print(f"Text-only Accuracy: {accuracy_score(true_labels, text_pred):.4f}")
print(f"Text-only Weighted F1: {f1_score(true_labels, text_pred, average='weighted'):.4f}")
print(f"Vision-only Accuracy: {accuracy_score(true_labels, vision_pred):.4f}")
print(f"Vision-only Weighted F1: {f1_score(true_labels, vision_pred, average='weighted'):.4f}")

# ============================================================
# 10. Summary Table
# ============================================================
print("\n" + "=" * 60)
print("CH-SIMS SUMMARY TABLE (10 Epochs)")
print("=" * 60)
print(f"{'Method':<35} {'Accuracy':<12} {'Weighted F1':<12}")
print("-" * 60)
print(f"{'Transformer + Dense (Late Fusion)':<35} {accuracy_score(true_labels, predictions):.4f}      {f1_score(true_labels, predictions, average='weighted'):.4f}")
print(f"{'Text-only (Transformer)':<35} {accuracy_score(true_labels, text_pred):.4f}      {f1_score(true_labels, text_pred, average='weighted'):.4f}")
print(f"{'Vision-only (Dense)':<35} {accuracy_score(true_labels, vision_pred):.4f}      {f1_score(true_labels, vision_pred, average='weighted'):.4f}")
print("=" * 60)

# ============================================================
# 11. Runtime
# ============================================================
end_time = time.time()
elapsed_time = end_time - start_time
print(f"\nTotal Execution Time: {elapsed_time:.4f} seconds")

CH-SIMS Binary Classification (Transformer + Dense)
Train: Text (1161, 768), Vision (1161, 709)
Labels (train): [742 419]
Labels (test): [248 140]

Model Summary:
Model: "model_14"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 text_input (InputLayer)     [(None, 768)]                0         []                            
                                                                                                  
 reshape_1 (Reshape)         (None, 768, 1)               0         ['text_input[0][0]']          
                                                                                                  
 multi_head_attention_1 (Mu  (None, 768, 1)               897       ['reshape_1[0][0]',           
 ltiHeadAttention)                                                   'reshape_1[0][0]']           
                           

C:\Users\ausco\anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\ausco\anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\ausco\anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [11]:
"""
CH-SIMS: Dense + Dense (Late Fusion) - MOD
Paper-Ready Version - 10 Epochs for Fair Comparison
"""

import pickle
import time
import numpy as np
from tensorflow.keras.layers import (Input, Dense, Dropout, BatchNormalization)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import legacy
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import accuracy_score, f1_score, classification_report

start_time = time.time()

# ============================================================
# 1. Load CH-SIMS Data
# ============================================================
file_path = 'SIMS/Processed/unaligned_39.pkl'
with open(file_path, 'rb') as f:
    data = pickle.load(f)

train = data['train']
valid = data['valid']
test = data['test']

def aggregate_mean(X):
    return np.mean(X, axis=1)

X_text_train = aggregate_mean(train['text'])
X_vision_train = aggregate_mean(train['vision'])
y_train = train['classification_labels'].astype(int)

X_text_val = aggregate_mean(valid['text'])
X_vision_val = aggregate_mean(valid['vision'])
y_val = valid['classification_labels'].astype(int)

X_text_test = aggregate_mean(test['text'])
X_vision_test = aggregate_mean(test['vision'])
y_test = test['classification_labels'].astype(int)

# ============================================================
# 2. Convert to Binary Classification (Remove Neutral)
# ============================================================
def convert_to_binary(X_text, X_vision, y):
    mask = (y != 1)
    X_text_bin = X_text[mask]
    X_vision_bin = X_vision[mask]
    y_bin = y[mask]
    y_bin = np.where(y_bin == 2, 1, 0)
    return X_text_bin, X_vision_bin, y_bin

X_text_train_bin, X_vision_train_bin, y_train_bin = convert_to_binary(
    X_text_train, X_vision_train, y_train
)
X_text_val_bin, X_vision_val_bin, y_val_bin = convert_to_binary(
    X_text_val, X_vision_val, y_val
)
X_text_test_bin, X_vision_test_bin, y_test_bin = convert_to_binary(
    X_text_test, X_vision_test, y_test
)

print("=" * 60)
print("CH-SIMS Binary Classification (Dense + Dense - Late Fusion)")
print("=" * 60)
print(f"Train: Text {X_text_train_bin.shape}, Vision {X_vision_train_bin.shape}")
print(f"Labels (train): {np.bincount(y_train_bin)}")
print(f"Labels (test): {np.bincount(y_test_bin)}")
print("=" * 60)

# ============================================================
# 3. Convert labels to one-hot
# ============================================================
num_classes = 2
y_train_cat = to_categorical(y_train_bin, num_classes)
y_val_cat = to_categorical(y_val_bin, num_classes)
y_test_cat = to_categorical(y_test_bin, num_classes)

# ============================================================
# 4. Build Models
# ============================================================

# ----- Text Branch: Dense -----
def build_text_branch(input_dim):
    text_input = Input(shape=(input_dim,), name='text_input')
    x = Dense(256, activation='relu')(text_input)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.3)(x)
    x = Dense(64, activation='relu')(x)
    text_out = Dense(num_classes, activation='softmax', name='text_softmax')(x)
    return text_input, text_out

# ----- Vision Branch: Dense -----
def build_vision_branch(input_dim):
    vision_input = Input(shape=(input_dim,), name='vision_input')
    x = Dense(256, activation='relu')(vision_input)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.3)(x)
    x = Dense(64, activation='relu')(x)
    vision_out = Dense(num_classes, activation='softmax', name='vision_softmax')(x)
    return vision_input, vision_out

# ----- Build Model -----
text_input, text_out = build_text_branch(X_text_train_bin.shape[1])
vision_input, vision_out = build_vision_branch(X_vision_train_bin.shape[1])

model = Model(
    inputs=[text_input, vision_input],
    outputs=[text_out, vision_out]
)

# ============================================================
# 5. Compile Model
# ============================================================
optimizer = legacy.Adam(learning_rate=0.001)
model.compile(
    loss='categorical_crossentropy',
    optimizer=optimizer,
    loss_weights={'text_softmax': 1.0, 'vision_softmax': 1.0},
    metrics={'text_softmax': ['accuracy'], 'vision_softmax': ['accuracy']}
)

print("\nModel Summary:")
model.summary()

# ============================================================
# 6. Training (10 Epochs)
# ============================================================
EPOCHS = 10
BATCH_SIZE = 64

print("\n" + "=" * 60)
print(f"Training Model ({EPOCHS} epochs, batch_size={BATCH_SIZE})...")
print("=" * 60)

history = model.fit(
    [X_text_train_bin, X_vision_train_bin],
    [y_train_cat, y_train_cat],
    validation_data=([X_text_val_bin, X_vision_val_bin], [y_val_cat, y_val_cat]),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1
)

# ============================================================
# 7. Prediction & Late Fusion (Average)
# ============================================================
print("\nPredicting...")
text_probs, vision_probs = model.predict([X_text_test_bin, X_vision_test_bin])

# Late Fusion: Average
ensemble_probs = (text_probs + vision_probs) / 2.0
predictions = np.argmax(ensemble_probs, axis=1)
true_labels = y_test_bin

# ============================================================
# 8. Evaluation (Weighted F1)
# ============================================================
print("\n" + "=" * 60)
print("CH-SIMS RESULTS (Dense + Dense - Late Fusion)")
print("=" * 60)
print(f"Test Accuracy: {accuracy_score(true_labels, predictions):.4f}")
print(f"Test Weighted F1: {f1_score(true_labels, predictions, average='weighted'):.4f}")
print("\nTest Classification Report:")
print(classification_report(
    true_labels, predictions, digits=4,
    target_names=['negative', 'positive']
))

# ============================================================
# 9. Unimodal Baselines
# ============================================================
print("\n" + "=" * 60)
print("UNIMODAL BASELINES")
print("=" * 60)

text_pred = np.argmax(text_probs, axis=1)
vision_pred = np.argmax(vision_probs, axis=1)

print(f"Text-only Accuracy: {accuracy_score(true_labels, text_pred):.4f}")
print(f"Text-only Weighted F1: {f1_score(true_labels, text_pred, average='weighted'):.4f}")
print(f"Vision-only Accuracy: {accuracy_score(true_labels, vision_pred):.4f}")
print(f"Vision-only Weighted F1: {f1_score(true_labels, vision_pred, average='weighted'):.4f}")

# ============================================================
# 10. Summary Table
# ============================================================
print("\n" + "=" * 60)
print("CH-SIMS SUMMARY TABLE (10 Epochs)")
print("=" * 60)
print(f"{'Method':<35} {'Accuracy':<12} {'Weighted F1':<12}")
print("-" * 60)
print(f"{'Dense + Dense (Late Fusion)':<35} {accuracy_score(true_labels, predictions):.4f}      {f1_score(true_labels, predictions, average='weighted'):.4f}")
print(f"{'Text-only (Dense)':<35} {accuracy_score(true_labels, text_pred):.4f}      {f1_score(true_labels, text_pred, average='weighted'):.4f}")
print(f"{'Vision-only (Dense)':<35} {accuracy_score(true_labels, vision_pred):.4f}      {f1_score(true_labels, vision_pred, average='weighted'):.4f}")
print("=" * 60)


CH-SIMS Binary Classification (Dense + Dense - Late Fusion)
Train: Text (1161, 768), Vision (1161, 709)
Labels (train): [742 419]
Labels (test): [248 140]

Model Summary:
Model: "model_15"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 text_input (InputLayer)     [(None, 768)]                0         []                            
                                                                                                  
 vision_input (InputLayer)   [(None, 709)]                0         []                            
                                                                                                  
 dense_59 (Dense)            (None, 256)                  196864    ['text_input[0][0]']          
                                                                                                  
 dense_62 (Dense)  


Predicting...
13/13 [==============================] - 0s 2ms/step

CH-SIMS RESULTS (Dense + Dense - Late Fusion)
Test Accuracy: 0.7964
Test Weighted F1: 0.7932

Test Classification Report:
              precision    recall  f1-score   support

    negative     0.8189    0.8750    0.8460       248
    positive     0.7480    0.6571    0.6996       140

    accuracy                         0.7964       388
   macro avg     0.7834    0.7661    0.7728       388
weighted avg     0.7933    0.7964    0.7932       388


UNIMODAL BASELINES
Text-only Accuracy: 0.7629
Text-only Weighted F1: 0.7648
Vision-only Accuracy: 0.6392
Vision-only Weighted F1: 0.4985

CH-SIMS SUMMARY TABLE (10 Epochs)
Method                              Accuracy     Weighted F1 
------------------------------------------------------------
Dense + Dense (Late Fusion)         0.7964      0.7932
Text-only (Dense)                   0.7629      0.7648
Vision-only (Dense)                 0.6392      0.4985


# Early Fusion

In [12]:
"""
CH-SIMS: Transformer for Text + Dense for Vision (Early Fusion)
Paper-Ready Version - 10 Epochs for Fair Comparison
"""

import pickle
import time
import numpy as np
from tensorflow.keras.layers import (Input, Dense, Dropout, BatchNormalization, 
                                     MultiHeadAttention, LayerNormalization,
                                     GlobalAveragePooling1D, Concatenate, Reshape)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import legacy
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import accuracy_score, f1_score, classification_report

start_time = time.time()

# ============================================================
# 1. Load CH-SIMS Data
# ============================================================
file_path = 'SIMS/Processed/unaligned_39.pkl'
with open(file_path, 'rb') as f:
    data = pickle.load(f)

train = data['train']
valid = data['valid']
test = data['test']

def aggregate_mean(X):
    return np.mean(X, axis=1)

X_text_train = aggregate_mean(train['text'])
X_vision_train = aggregate_mean(train['vision'])
y_train = train['classification_labels'].astype(int)

X_text_val = aggregate_mean(valid['text'])
X_vision_val = aggregate_mean(valid['vision'])
y_val = valid['classification_labels'].astype(int)

X_text_test = aggregate_mean(test['text'])
X_vision_test = aggregate_mean(test['vision'])
y_test = test['classification_labels'].astype(int)

# ============================================================
# 2. Convert to Binary Classification (Remove Neutral)
# ============================================================
def convert_to_binary(X_text, X_vision, y):
    mask = (y != 1)
    X_text_bin = X_text[mask]
    X_vision_bin = X_vision[mask]
    y_bin = y[mask]
    y_bin = np.where(y_bin == 2, 1, 0)
    return X_text_bin, X_vision_bin, y_bin

X_text_train_bin, X_vision_train_bin, y_train_bin = convert_to_binary(
    X_text_train, X_vision_train, y_train
)
X_text_val_bin, X_vision_val_bin, y_val_bin = convert_to_binary(
    X_text_val, X_vision_val, y_val
)
X_text_test_bin, X_vision_test_bin, y_test_bin = convert_to_binary(
    X_text_test, X_vision_test, y_test
)

print("=" * 60)
print("CH-SIMS Binary Classification (Transformer + Dense - Early Fusion)")
print("=" * 60)
print(f"Train: Text {X_text_train_bin.shape}, Vision {X_vision_train_bin.shape}")
print(f"Labels (train): {np.bincount(y_train_bin)}")
print(f"Labels (test): {np.bincount(y_test_bin)}")
print("=" * 60)

# ============================================================
# 3. Convert labels to one-hot
# ============================================================
num_classes = 2
y_train_cat = to_categorical(y_train_bin, num_classes)
y_val_cat = to_categorical(y_val_bin, num_classes)
y_test_cat = to_categorical(y_test_bin, num_classes)

# ============================================================
# 4. Build Model (Early Fusion)
# ============================================================

# ----- Text Branch: Transformer -----
text_input = Input(shape=(X_text_train_bin.shape[1],), name='text_input')
x_text = Reshape((X_text_train_bin.shape[1], 1))(text_input)
x_text = MultiHeadAttention(num_heads=4, key_dim=32)(x_text, x_text)
x_text = LayerNormalization()(x_text)
x_text = GlobalAveragePooling1D()(x_text)

# ----- Vision Branch: Dense -----
vision_input = Input(shape=(X_vision_train_bin.shape[1],), name='vision_input')
x_vision = Dense(128, activation='relu')(vision_input)
x_vision = BatchNormalization()(x_vision)
x_vision = Dropout(0.3)(x_vision)
x_vision = Dense(64, activation='relu')(x_vision)
x_vision = Dropout(0.3)(x_vision)

# ----- Early Fusion: Concatenate -----
concatenated = Concatenate()([x_text, x_vision])

# ----- Fusion Layers -----
fusion = Dense(64, activation='relu')(concatenated)
fusion = BatchNormalization()(fusion)
fusion = Dropout(0.3)(fusion)
fusion = Dense(64, activation='relu')(fusion)
fusion = Dropout(0.3)(fusion)
output = Dense(num_classes, activation='softmax')(fusion)

# ----- Model -----
model = Model(inputs=[text_input, vision_input], outputs=output)

# ============================================================
# 5. Compile Model
# ============================================================
optimizer = legacy.Adam(learning_rate=0.001)
model.compile(
    loss='categorical_crossentropy',
    optimizer=optimizer,
    metrics=['accuracy']
)

print("\nModel Summary:")
model.summary()

# ============================================================
# 6. Training (10 Epochs for Fair Comparison)
# ============================================================
EPOCHS = 10
BATCH_SIZE = 64

print("\n" + "=" * 60)
print(f"Training Model ({EPOCHS} epochs, batch_size={BATCH_SIZE})...")
print("=" * 60)

history = model.fit(
    [X_text_train_bin, X_vision_train_bin],
    y_train_cat,
    validation_data=([X_text_val_bin, X_vision_val_bin], y_val_cat),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1
)

# ============================================================
# 7. Prediction & Evaluation
# ============================================================
print("\nPredicting...")
predictions_prob = model.predict([X_text_test_bin, X_vision_test_bin])
predictions = np.argmax(predictions_prob, axis=1)
true_labels = y_test_bin

# ============================================================
# 8. Evaluation (Weighted F1)
# ============================================================
print("\n" + "=" * 60)
print("CH-SIMS RESULTS (Transformer + Dense - Early Fusion)")
print("=" * 60)
print(f"Test Accuracy: {accuracy_score(true_labels, predictions):.4f}")
print(f"Test Weighted F1: {f1_score(true_labels, predictions, average='weighted'):.4f}")
print("\nTest Classification Report:")
print(classification_report(
    true_labels, predictions, digits=4,
    target_names=['negative', 'positive']
))

# ============================================================
# 9. Summary Table
# ============================================================
print("\n" + "=" * 60)
print("CH-SIMS SUMMARY TABLE (10 Epochs)")
print("=" * 60)
print(f"{'Method':<35} {'Accuracy':<12} {'Weighted F1':<12}")
print("-" * 60)
print(f"{'Transformer + Dense (Early Fusion)':<35} {accuracy_score(true_labels, predictions):.4f}      {f1_score(true_labels, predictions, average='weighted'):.4f}")
print("=" * 60)

# ============================================================
# 10. Runtime
# ============================================================
end_time = time.time()
elapsed_time = end_time - start_time
print(f"\nTotal Execution Time: {elapsed_time:.4f} seconds")

CH-SIMS Binary Classification (Transformer + Dense - Early Fusion)
Train: Text (1161, 768), Vision (1161, 709)
Labels (train): [742 419]
Labels (test): [248 140]

Model Summary:
Model: "model_16"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 vision_input (InputLayer)   [(None, 709)]                0         []                            
                                                                                                  
 text_input (InputLayer)     [(None, 768)]                0         []                            
                                                                                                  
 dense_65 (Dense)            (None, 128)                  90880     ['vision_input[0][0]']        
                                                                                                  
 reshape_2 (

In [13]:
"""
CH-SIMS: Dense + Dense (Early Fusion)
Paper-Ready Version - 10 Epochs for Fair Comparison
Consistent with CH-SIMS experimental setup
"""

import pickle
import time
import numpy as np
from tensorflow.keras.layers import (Input, Dense, Dropout, BatchNormalization, Concatenate)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import legacy
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import accuracy_score, f1_score, classification_report

start_time = time.time()

# ============================================================
# 1. Load CH-SIMS Data
# ============================================================
file_path = 'SIMS/Processed/unaligned_39.pkl'
with open(file_path, 'rb') as f:
    data = pickle.load(f)

train = data['train']
valid = data['valid']
test = data['test']

def aggregate_mean(X):
    return np.mean(X, axis=1)

X_text_train = aggregate_mean(train['text'])
X_vision_train = aggregate_mean(train['vision'])
y_train = train['classification_labels'].astype(int)

X_text_val = aggregate_mean(valid['text'])
X_vision_val = aggregate_mean(valid['vision'])
y_val = valid['classification_labels'].astype(int)

X_text_test = aggregate_mean(test['text'])
X_vision_test = aggregate_mean(test['vision'])
y_test = test['classification_labels'].astype(int)

# ============================================================
# 2. Convert to Binary Classification (Remove Neutral)
# ============================================================
def convert_to_binary(X_text, X_vision, y):
    mask = (y != 1)
    X_text_bin = X_text[mask]
    X_vision_bin = X_vision[mask]
    y_bin = y[mask]
    y_bin = np.where(y_bin == 2, 1, 0)
    return X_text_bin, X_vision_bin, y_bin

X_text_train_bin, X_vision_train_bin, y_train_bin = convert_to_binary(
    X_text_train, X_vision_train, y_train
)
X_text_val_bin, X_vision_val_bin, y_val_bin = convert_to_binary(
    X_text_val, X_vision_val, y_val
)
X_text_test_bin, X_vision_test_bin, y_test_bin = convert_to_binary(
    X_text_test, X_vision_test, y_test
)

print("=" * 60)
print("CH-SIMS Binary Classification (Dense + Dense - Early Fusion)")
print("=" * 60)
print(f"Train: Text {X_text_train_bin.shape}, Vision {X_vision_train_bin.shape}")
print(f"Labels (train): {np.bincount(y_train_bin)}")
print(f"Labels (test): {np.bincount(y_test_bin)}")
print("=" * 60)

# ============================================================
# 3. Convert labels to one-hot
# ============================================================
num_classes = 2
y_train_cat = to_categorical(y_train_bin, num_classes)
y_val_cat = to_categorical(y_val_bin, num_classes)
y_test_cat = to_categorical(y_test_bin, num_classes)

# ============================================================
# 4. Build Model (Early Fusion)
# ============================================================

# ----- Text Branch: Dense -----
text_input = Input(shape=(X_text_train_bin.shape[1],), name='text_input')
x_text = Dense(256, activation='relu')(text_input)
x_text = BatchNormalization()(x_text)
x_text = Dropout(0.3)(x_text)
x_text = Dense(128, activation='relu')(x_text)
x_text = Dropout(0.3)(x_text)
x_text = Dense(64, activation='relu')(x_text)

# ----- Vision Branch: Dense -----
vision_input = Input(shape=(X_vision_train_bin.shape[1],), name='vision_input')
x_vision = Dense(256, activation='relu')(vision_input)
x_vision = BatchNormalization()(x_vision)
x_vision = Dropout(0.3)(x_vision)
x_vision = Dense(128, activation='relu')(x_vision)
x_vision = Dropout(0.3)(x_vision)
x_vision = Dense(64, activation='relu')(x_vision)

# ----- Early Fusion: Concatenate -----
concatenated = Concatenate()([x_text, x_vision])

# ----- Fusion Layers -----
fusion = Dense(128, activation='relu')(concatenated)
fusion = BatchNormalization()(fusion)
fusion = Dropout(0.5)(fusion)
fusion = Dense(64, activation='relu')(fusion)
fusion = Dropout(0.3)(fusion)
output = Dense(num_classes, activation='softmax')(fusion)

# ----- Model -----
model = Model(inputs=[text_input, vision_input], outputs=output)

# ============================================================
# 5. Compile Model
# ============================================================
optimizer = legacy.Adam(learning_rate=0.001)
model.compile(
    loss='categorical_crossentropy',
    optimizer=optimizer,
    metrics=['accuracy']
)

print("\nModel Summary:")
model.summary()

# ============================================================
# 6. Training (10 Epochs)
# ============================================================
EPOCHS = 10
BATCH_SIZE = 64

print("\n" + "=" * 60)
print(f"Training Model ({EPOCHS} epochs, batch_size={BATCH_SIZE})...")
print("=" * 60)

history = model.fit(
    [X_text_train_bin, X_vision_train_bin],
    y_train_cat,
    validation_data=([X_text_val_bin, X_vision_val_bin], y_val_cat),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1
)

# ============================================================
# 7. Prediction & Evaluation
# ============================================================
print("\nPredicting...")
predictions_prob = model.predict([X_text_test_bin, X_vision_test_bin])
predictions = np.argmax(predictions_prob, axis=1)
true_labels = y_test_bin

# ============================================================
# 8. Evaluation (Weighted F1)
# ============================================================
print("\n" + "=" * 60)
print("CH-SIMS RESULTS (Dense + Dense - Early Fusion)")
print("=" * 60)
print(f"Test Accuracy: {accuracy_score(true_labels, predictions):.4f}")
print(f"Test Weighted F1: {f1_score(true_labels, predictions, average='weighted'):.4f}")
print("\nTest Classification Report:")
print(classification_report(
    true_labels, predictions, digits=4,
    target_names=['negative', 'positive']
))

# ============================================================
# 9. Summary Table
# ============================================================
print("\n" + "=" * 60)
print("CH-SIMS SUMMARY TABLE (10 Epochs)")
print("=" * 60)
print(f"{'Method':<35} {'Accuracy':<12} {'Weighted F1':<12}")
print("-" * 60)
print(f"{'Dense + Dense (Early Fusion)':<35} {accuracy_score(true_labels, predictions):.4f}      {f1_score(true_labels, predictions, average='weighted'):.4f}")
print("=" * 60)

# ============================================================
# 10. Runtime
# ============================================================
end_time = time.time()
elapsed_time = end_time - start_time
print(f"\nTotal Execution Time: {elapsed_time:.4f} seconds")

CH-SIMS Binary Classification (Dense + Dense - Early Fusion)
Train: Text (1161, 768), Vision (1161, 709)
Labels (train): [742 419]
Labels (test): [248 140]

Model Summary:
Model: "model_17"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 text_input (InputLayer)     [(None, 768)]                0         []                            
                                                                                                  
 vision_input (InputLayer)   [(None, 709)]                0         []                            
                                                                                                  
 dense_70 (Dense)            (None, 256)                  196864    ['text_input[0][0]']          
                                                                                                  
 dense_73 (Dense) 